In [5]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pickle
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

In [6]:
env = gym.make('FrozenLake-v1', map_name = '4x4', is_slippery = False, render_mode = 'rgb_array')
q_table = np.zeros(shape=(env.observation_space.n, env.action_space.n))
rewards_per_episode = []
epsilon_decay_graph = []

In [13]:
learning_rate = 0.9 # alpha
discount_factor = 0.9 # gamma
epsilon = 0.9 # random actions
epsilon_decay_rate = 0.0001
n_episodes = 6000

In [14]:
# Monte Carlo
for i in range(n_episodes):

    terminated, truncated = False, False
    states, rewards, actions = [], [], []
    state, info = env.reset()

    
    # The while loop below get the sequence of States in the episode
    while (not terminated and not truncated):

        states.append(state)
        
        if (np.random.random() < epsilon): # Exploaratory action 90% of the time
                action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        next_state, reward, terminated, truncated, _ = env.step(action)

        rewards.append(reward)
        actions.append(action)
        state = next_state
    
    T = len(states)
    returns = []

    for i in range(T):
        G_t = 0
        for j in range(i,T):
            G_t += (discount_factor ** (j - i)) * rewards[j]
        returns.append(G_t)
    
    # Updating the q_values for use in the next episode
    for t in range(T):
        s_t = states[t]
        a_t = actions[t]
        G_t = returns[t]
        
        q_table[s_t][a_t] += learning_rate * (G_t - q_table[s_t][a_t])
    epsilon = max(0.01, epsilon * 0.995)

In [15]:
f = open('frozen_lake4x4.pk1', 'wb')
pickle.dump(q_table,f) # Saving the Q_Table
f.close()


# view the returns list normally
with open('frozen_lake4x4.pk1', 'rb') as f:
    loaded_q_table = pickle.load(f)
print(loaded_q_table)

[[4.78296900e-001 5.90490000e-001 4.31057700e-001 5.09677960e-101]
 [4.83079869e-001 0.00000000e+000 6.56100000e-002 5.31441000e-100]
 [5.31441000e-098 7.29000000e-002 5.31441000e-050 5.90490000e-001]
 [5.90490000e-050 0.00000000e+000 0.00000000e+000 0.00000000e+000]
 [5.84585100e-100 6.56100000e-001 0.00000000e+000 4.78296900e-063]
 [0.00000000e+000 0.00000000e+000 0.00000000e+000 0.00000000e+000]
 [0.00000000e+000 8.10000000e-001 0.00000000e+000 6.49539000e-001]
 [0.00000000e+000 0.00000000e+000 0.00000000e+000 0.00000000e+000]
 [6.49539000e-001 0.00000000e+000 7.29000000e-001 5.79270690e-050]
 [5.90490000e-001 7.37100000e-001 8.10000000e-001 0.00000000e+000]
 [7.13928183e-001 9.00000000e-001 0.00000000e+000 7.27024410e-001]
 [0.00000000e+000 0.00000000e+000 0.00000000e+000 0.00000000e+000]
 [0.00000000e+000 0.00000000e+000 0.00000000e+000 0.00000000e+000]
 [0.00000000e+000 8.01900000e-001 9.00000000e-001 7.28861562e-001]
 [8.10000000e-001 9.00000000e-001 1.00000000e+000 8.08462539e-